# L9d: Skip-Gram Word Embeddings and Negative Sampling
In this lab, we implement and compare two training approaches for the Skip-Gram model: full softmax and negative sampling. We then compare the embeddings learned by Skip-Gram to those from CBOW on the same toy corpus.

> __Learning Objectives__
> * __Build Skip-Gram training pairs:__ Generate (target, context) training pairs and compare the number of pairs produced by Skip-Gram versus CBOW for the same window size.
> * __Train Skip-Gram with full softmax and negative sampling:__ Implement both training objectives and compare loss convergence across the two approaches.
> * __Compare embeddings across models:__ Use cosine similarity to compare word associations captured by CBOW, Skip-Gram softmax, and Skip-Gram negative sampling.

___

## Setup, Data, and Prerequisites

> We load packages and source files via `Include.jl`. This file loads `VLDataScienceMachineLearningPackage.jl` along with local implementations of CBOW, Skip-Gram softmax, and Skip-Gram negative sampling.

In [ ]:
include("Include.jl")

### Implementations

| Function | Location | Description |
|----------|----------|-------------|
| `build_cbow_pairs(sentences, vocabulary; window_size)` | `src/CBOW.jl` | Generates (context sum, target) pairs for CBOW |
| `train_cbow(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/CBOW.jl` | Trains CBOW via backpropagation |
| `build_skipgram_pairs(sentences, vocabulary; window_size)` | `src/SkipGram.jl` | Generates one (target, context) pair per target-context word pair |
| `train_skipgram_softmax(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/SkipGram.jl` | Trains Skip-Gram with full softmax |
| `build_noise_distribution(sentences, vocabulary)` | `src/NegativeSampling.jl` | Builds smoothed unigram noise distribution $P(w) \propto f(w)^{0.75}$ |
| `train_skipgram_ns(training_pairs, vocab_size, noise_dist; d_h, eta, num_epochs, k)` | `src/NegativeSampling.jl` | Trains Skip-Gram with negative sampling |

### Constants

In [ ]:
sentences = [
    "I love julia and machine learning",
    "julia is fun and great",
    "I enjoy data science and machine learning",
    "data science is great and fun",
    "machine learning is a great and fun subject",
    "I love learning new things about data science"
];

control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];

window_size = 2;   # context window half-width
d_h = 5;           # embedding dimension
eta = 0.01;        # learning rate
num_epochs = 500;  # training epochs
k = 5;             # number of negative samples per positive pair

### Data

In [ ]:
all_words = vcat([split(lowercase(s)) for s in sentences]...)
unique_words = unique(all_words)
vocabulary = Dict{String, Int64}()
for (i, token) in enumerate(control_tokens)
    vocabulary[token] = i
end
offset = length(control_tokens)
for (i, word) in enumerate(unique_words)
    if !haskey(vocabulary, word)
        vocabulary[word] = offset + i
    end
end
inverse_vocabulary = Dict{Int64, String}(v => k for (k, v) in vocabulary)
println("Vocabulary size: $(length(vocabulary))")

___

## Task 1: Build Skip-Gram Training Pairs

> In Skip-Gram, the model predicts context words from a target word. For each target word at position $i$, we create one (target, context) pair for every context word within the window $[i - m, i + m]$, where $m$ is the window half-width. This produces $2m$ training pairs per word occurrence, compared to one pair per word in CBOW. We call `build_skipgram_pairs` to generate these pairs and compare the count to CBOW pairs.

In [ ]:
skipgram_pairs = build_skipgram_pairs(sentences, vocabulary; window_size=window_size);
cbow_pairs = build_cbow_pairs(sentences, vocabulary; window_size=window_size);
println("Skip-Gram training pairs: $(length(skipgram_pairs))")
println("CBOW training pairs:      $(length(cbow_pairs))")
println("Ratio (SG/CBOW):          $(round(length(skipgram_pairs)/length(cbow_pairs), digits=2))")

__Discussion:__ Why does Skip-Gram produce more training pairs than CBOW for the same window size? How would doubling the window size affect the ratio? What is the computational implication for training?

___

## Task 2: Train Skip-Gram with Full Softmax

> Skip-Gram with full softmax uses cross-entropy loss over the entire vocabulary for each (target, context) pair. At each step we compute $\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}_2 \mathbf{W}_1 \mathbf{x})$ and minimize $-\log \hat{y}_{c}$, where $c$ is the index of the true context word. The cost per update is $O(|V|)$ because the softmax normalization sums over all vocabulary entries.

In [ ]:
Random.seed!(42)
W1_sg, W2_sg, loss_sg = train_skipgram_softmax(skipgram_pairs, length(vocabulary);
    d_h=d_h, eta=eta, num_epochs=num_epochs);

plot(loss_sg; xlabel="Epoch", ylabel="Average Cross-Entropy Loss",
    title="Skip-Gram (Full Softmax) Training Loss", legend=false, lw=2)

___

## Task 3: Train Skip-Gram with Negative Sampling

> Negative Sampling replaces the full softmax with a binary classification task. For each positive (target, context) pair, we sample $k$ noise words from the smoothed unigram distribution $P_n(w) \propto f(w)^{3/4}$ and train the model to score the positive pair higher than the noise pairs using sigmoid activations. The objective for one training example is:
> $$\mathcal{L} = -\log \sigma(\mathbf{w}_{c}^{\top} \mathbf{h}) - \sum_{i=1}^{k} \log \sigma(-\mathbf{w}_{n_i}^{\top} \mathbf{h})$$
> where $\mathbf{h} = \mathbf{W}_1[:,t]$ is the target embedding. This reduces the per-update cost from $O(|V|)$ to $O(k)$.

In [ ]:
noise_distribution = build_noise_distribution(sentences, vocabulary);

Random.seed!(42)
W1_ns, W2_ns, loss_ns = train_skipgram_ns(skipgram_pairs, length(vocabulary), noise_distribution;
    d_h=d_h, eta=eta, num_epochs=num_epochs, k=k);

plot(loss_ns; xlabel="Epoch", ylabel="Average Loss",
    title="Skip-Gram (Negative Sampling) Training Loss", legend=false, lw=2)

We plot both loss curves together for comparison.

In [ ]:
plot(loss_sg; label="SG Softmax", lw=2)
plot!(loss_ns; label="SG Neg. Sampling", lw=2, linestyle=:dash)
xlabel!("Epoch")
ylabel!("Average Loss")
title!("Loss Comparison: Softmax vs. Negative Sampling")

__Discussion:__ The two loss curves use different objective functions (cross-entropy vs. binary cross-entropy), so their magnitudes are not directly comparable. What does each curve tell you about convergence? Which training approach would scale better to a vocabulary of 100,000 words?

___

## Task 4: Compare Embeddings Across Models

> We train CBOW on the same corpus and compare cosine similarities for selected word pairs across all three models: CBOW, Skip-Gram softmax, and Skip-Gram negative sampling. Embeddings are taken from the columns of $\mathbf{W}_1$ in each model.

In [ ]:
Random.seed!(42)
W1_cbow, _, _ = train_cbow(cbow_pairs, length(vocabulary); d_h=d_h, eta=eta, num_epochs=num_epochs);

In [ ]:
word_pairs = [
    ("machine", "learning"),
    ("data", "science"),
    ("julia", "fun"),
    ("love", "enjoy"),
    ("great", "fun"),
    ("machine", "julia"),
]

cosine_sim(W, w1, w2, vocab) = begin
    i = get(vocab, w1, 0)
    j = get(vocab, w2, 0)
    (i == 0 || j == 0) && return NaN
    e1 = W[:, i]; e2 = W[:, j]
    dot(e1, e2) / (norm(e1) * norm(e2) + 1e-10)
end

header = ["Word 1", "Word 2", "CBOW", "SG Softmax", "SG Neg. Samp."]
rows = []
for (w1, w2) in word_pairs
    c  = round(cosine_sim(W1_cbow, w1, w2, vocabulary), digits=3)
    sg = round(cosine_sim(W1_sg,   w1, w2, vocabulary), digits=3)
    ns = round(cosine_sim(W1_ns,   w1, w2, vocabulary), digits=3)
    push!(rows, [w1, w2, c, sg, ns])
end
pretty_table(hcat(rows...)'; header=header)

__Discussion:__ Do the three models agree on the most semantically similar pairs? Where do they disagree most? Given that the corpus has only 6 sentences, what would you expect to change if you trained on a larger corpus?

___

## Summary

In this lab, we trained Skip-Gram embeddings using both full softmax and negative sampling on a toy corpus, and compared the resulting embeddings to CBOW. Skip-Gram generates more training signal per word than CBOW, while negative sampling reduces the per-update cost from $O(|V|)$ to $O(k)$.

> __Key Takeaways__
> * **Skip-Gram generates more training pairs than CBOW:** For window size $m$, each word occurrence produces up to $2m$ (target, context) pairs, giving Skip-Gram more training signal per sentence than CBOW.
> * **Negative sampling replaces full softmax with binary classification:** Instead of normalizing over the full vocabulary, negative sampling trains the model to distinguish true context words from $k$ noise samples, reducing per-update cost from $O(|V|)$ to $O(k)$.
> * **Embedding quality depends on corpus size and model choice:** On small corpora, all three models (CBOW, SG-Softmax, SG-NegSamp) may produce similar or inconsistent similarities; differences become more pronounced and meaningful with larger training corpora.